# 02 - EDA Storytelling Star Wars BI

Este notebook es el analisis exploratorio orientado al dashboard. Parte de los CSV limpios generados por `01_limpieza_star_wars.ipynb` y organiza la lectura segun el nuevo storytelling:

**Choose Your Side - Star Wars Rebellion Lab**

Objetivo: descubrir que experiencia Star Wars debe recibir cada tipo de audiencia para reactivar su conexion emocional con la marca.


## 0. Configuracion y carga de tablas limpias


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


def find_project_root(start_path=None):
    start_path = Path(start_path or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    return start_path

BASE_DIR = find_project_root()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DOCS_DIR = BASE_DIR / "docs"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("Proyecto:", BASE_DIR)
print("Processed:", PROCESSED_DIR)


In [ ]:
# Tablas limpias generadas por 01_limpieza_star_wars.ipynb
survey_respondents = pd.read_csv(PROCESSED_DIR / "survey_respondents.csv")
survey_movies_seen = pd.read_csv(PROCESSED_DIR / "survey_movies_seen.csv")
survey_movie_rankings = pd.read_csv(PROCESSED_DIR / "survey_movie_rankings.csv")
survey_character_opinions = pd.read_csv(PROCESSED_DIR / "survey_character_opinions.csv")

universe_characters = pd.read_csv(PROCESSED_DIR / "universe_characters_clean.csv")
universe_films = pd.read_csv(PROCESSED_DIR / "universe_films_clean.csv")
universe_planets = pd.read_csv(PROCESSED_DIR / "universe_planets_clean.csv")
universe_starships = pd.read_csv(PROCESSED_DIR / "universe_starships_clean.csv")
universe_weapons = pd.read_csv(PROCESSED_DIR / "universe_weapons_clean.csv")
universe_quotes = pd.read_csv(PROCESSED_DIR / "universe_quotes_clean.csv")
universe_assets = pd.read_csv(PROCESSED_DIR / "universe_assets.csv")
universe_quality_summary = pd.read_csv(PROCESSED_DIR / "universe_quality_summary.csv")
films_business_clean = pd.read_csv(PROCESSED_DIR / "films_business_clean.csv", parse_dates=["release_date"])

print("Encuesta respondents:", survey_respondents.shape)
print("Opiniones personajes:", survey_character_opinions.shape)
print("Peliculas negocio:", films_business_clean.shape)
print("Activos universo:", universe_assets.shape)


In [ ]:
def missing_report(df):
    return (
        pd.DataFrame({
            "nulos": df.isna().sum(),
            "porcentaje": (df.isna().mean() * 100).round(2),
        })
        .query("nulos > 0")
        .sort_values("porcentaje", ascending=False)
    )


def split_keys(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def collect_list_keys(df, column):
    if column not in df.columns:
        return set()
    keys = set()
    for value in df[column].dropna():
        keys.update(split_keys(value))
    return keys


def export_csv(df, filename):
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print("Exportado:", filename, df.shape)
    return df


## Storytelling del informe

1. **La senal perdida:** localizar donde sigue viva la conexion con Star Wars.
2. **Los clanes de la galaxia:** segmentar la audiencia en tipos accionables.
3. **El mapa emocional de Star Wars:** interpretar personajes como emociones de marca.
4. **Las puertas de entrada al universo:** decidir que peliculas abren mejor la conversacion.
5. **Planetas como experiencias:** convertir mundos en atmosferas de campana.
6. **Tecnologia, poder y velocidad:** agrupar naves, vehiculos y armas como activos de espectaculo.
7. **La estrategia de reactivacion:** recomendar rutas de experiencia por tipo de publico.


In [ ]:
# Episodio I: El mercado galactico

survey_kpis = pd.DataFrame([{
    "respondents": len(survey_respondents),
    "seen_any_star_wars_pct": round(survey_respondents["has_seen_any_star_wars_film_binary"].mean() * 100, 2),
    "star_wars_fan_pct": round(survey_respondents["is_star_wars_fan_binary"].mean() * 100, 2),
    "avg_movies_seen": round(survey_respondents["total_movies_seen"].mean(), 2),
    "complete_movie_ranking_pct": round(survey_respondents["has_complete_movie_ranking"].mean() * 100, 2),
    "with_demographic_info_pct": round(survey_respondents["has_demographic_info"].mean() * 100, 2),
}])

fan_by_age = (
    survey_respondents.dropna(subset=["age"])
    .groupby("age", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_age["avg_movies_seen"] = fan_by_age["avg_movies_seen"].round(2)

fan_by_gender = (
    survey_respondents.dropna(subset=["gender"])
    .groupby("gender", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_gender["avg_movies_seen"] = fan_by_gender["avg_movies_seen"].round(2)

display(survey_kpis)
display(fan_by_age)
display(fan_by_gender)


In [ ]:
# Episodio II: La pelicula que abre el portal

movie_views_summary = (
    survey_movies_seen
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(viewers=("has_seen_movie", "sum"), respondents=("respondent_id", "nunique"))
)
movie_views_summary["view_rate_pct"] = (movie_views_summary["viewers"] / movie_views_summary["respondents"] * 100).round(2)

movie_rank_summary = (
    survey_movie_rankings.dropna(subset=["movie_rank"])
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(
        avg_rank=("movie_rank", "mean"),
        median_rank=("movie_rank", "median"),
        ranking_responses=("respondent_id", "nunique"),
        first_place_votes=("movie_rank", lambda s: (s == 1).sum()),
    )
)
movie_rank_summary["avg_rank"] = movie_rank_summary["avg_rank"].round(2)
movie_rank_summary["first_place_pct"] = (movie_rank_summary["first_place_votes"] / movie_rank_summary["ranking_responses"] * 100).round(2)
movie_rank_summary["preference_score"] = (7 - movie_rank_summary["avg_rank"]).round(2)

movie_opportunities = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "first_place_pct"]],
    on="film_key",
    how="left",
)
movie_opportunities["movie_campaign_score"] = (
    movie_opportunities["view_rate_pct"] * 0.45
    + (movie_opportunities["preference_score"] / 6 * 100) * 0.45
    + movie_opportunities["first_place_pct"] * 0.10
).round(2)
movie_opportunities = movie_opportunities.sort_values("movie_campaign_score", ascending=False)

movie_audience_summary = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "ranking_responses", "first_place_pct"]],
    on="film_key",
    how="left",
)

eda_movie_commercial_audience_summary = films_business_clean.merge(movie_audience_summary, on="film_key", how="left")
eda_movie_commercial_audience_summary["movie_title"] = eda_movie_commercial_audience_summary["movie_title"].fillna(eda_movie_commercial_audience_summary["film_title"])
eda_movie_commercial_audience_summary["is_in_survey"] = eda_movie_commercial_audience_summary["viewers"].notna().astype(int)

best_campaign_movie = movie_opportunities.iloc[0]
best_box_office_movie = films_business_clean.sort_values("worldwide_box_office_usd", ascending=False).iloc[0]
best_roi_movie = films_business_clean.sort_values("roi", ascending=False).iloc[0]

display(movie_opportunities)
display(eda_movie_commercial_audience_summary)
print("Pelicula eje recomendada:", best_campaign_movie["movie_title"])
print("Mayor taquilla:", best_box_office_movie["film_title"])
print("Mayor ROI:", best_roi_movie["film_title"])


In [ ]:
# Episodio III: El consejo de personajes

character_opinion_summary = (
    survey_character_opinions
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(
        avg_opinion_score=("opinion_score", "mean"),
        opinion_responses=("opinion_label", lambda s: s.notna().sum()),
        favorable_responses=("opinion_score", lambda s: (s > 0).sum()),
        unfavorable_responses=("opinion_score", lambda s: (s < 0).sum()),
        unfamiliar_responses=("is_unfamiliar", "sum"),
        total_rows=("respondent_id", "count"),
    )
)
character_opinion_summary["avg_opinion_score"] = character_opinion_summary["avg_opinion_score"].round(3)
character_opinion_summary["favorable_pct"] = (character_opinion_summary["favorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfavorable_pct"] = (character_opinion_summary["unfavorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfamiliar_pct"] = (character_opinion_summary["unfamiliar_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary = character_opinion_summary.sort_values(["avg_opinion_score", "favorable_pct"], ascending=False)

quote_character_summary = (
    universe_quotes
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(quote_count=("quote", "count"))
    .sort_values("quote_count", ascending=False)
)

characters_for_merge = universe_characters[["character_key", "name", "species", "gender", "homeworld", "height", "weight"]].rename(columns={"name": "universe_character_name"})
asset_quality_for_merge = universe_assets[universe_assets["asset_type"] == "character"][["asset_key", "data_completeness_pct", "internal_presence_score"]].rename(columns={"asset_key": "character_key"})

character_opportunities = character_opinion_summary.merge(characters_for_merge, on="character_key", how="left")
character_opportunities = character_opportunities.merge(quote_character_summary[["character_key", "quote_count"]], on="character_key", how="left")
character_opportunities = character_opportunities.merge(asset_quality_for_merge, on="character_key", how="left")
character_opportunities["quote_count"] = character_opportunities["quote_count"].fillna(0)
character_opportunities["is_in_universe_dataset"] = character_opportunities["universe_character_name"].notna().astype(int)
character_opportunities["data_completeness_pct"] = character_opportunities["data_completeness_pct"].fillna(0)
character_opportunities["internal_presence_score"] = character_opportunities["internal_presence_score"].fillna(0) + character_opportunities["quote_count"]

character_opportunities["audience_affinity_score"] = (((character_opportunities["avg_opinion_score"] + 2) / 4) * 100).round(2)
character_opportunities["familiarity_score"] = (100 - character_opportunities["unfamiliar_pct"]).round(2)
max_presence = character_opportunities["internal_presence_score"].max()
character_opportunities["internal_presence_score_scaled"] = np.where(max_presence > 0, (character_opportunities["internal_presence_score"] / max_presence * 100).round(2), 0)
character_opportunities["data_quality_score"] = character_opportunities["data_completeness_pct"].round(2)
character_opportunities["merchandising_potential_index"] = (
    character_opportunities["audience_affinity_score"] * 0.45
    + character_opportunities["familiarity_score"] * 0.25
    + character_opportunities["internal_presence_score_scaled"] * 0.20
    + character_opportunities["data_quality_score"] * 0.10
).round(2)

presence_median = character_opportunities["internal_presence_score_scaled"].median()
audience_median = character_opportunities["audience_affinity_score"].median()
character_opportunities["opportunity_quadrant"] = np.select(
    [
        (character_opportunities["internal_presence_score_scaled"] >= presence_median) & (character_opportunities["audience_affinity_score"] >= audience_median),
        (character_opportunities["internal_presence_score_scaled"] >= presence_median) & (character_opportunities["audience_affinity_score"] < audience_median),
        (character_opportunities["internal_presence_score_scaled"] < presence_median) & (character_opportunities["audience_affinity_score"] >= audience_median),
    ],
    ["priority_campaign", "repositioning_needed", "hidden_opportunity"],
    default="low_priority",
)
character_opportunities = character_opportunities.sort_values("merchandising_potential_index", ascending=False)

display(character_opportunities.head(14))


In [ ]:
# Paginas 5 y 6: Planetas, tecnologia y activos visuales

universe_overview = pd.DataFrame([
    {"asset_type": "characters", "count": len(universe_characters)},
    {"asset_type": "films", "count": len(universe_films)},
    {"asset_type": "planets", "count": len(universe_planets)},
    {"asset_type": "starships", "count": len(universe_starships)},
    {"asset_type": "weapons", "count": len(universe_weapons)},
    {"asset_type": "quotes", "count": len(universe_quotes)},
])

character_species_summary = universe_characters.groupby("species", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)
character_gender_summary = universe_characters.groupby("gender", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)
character_homeworld_summary = universe_characters.groupby("homeworld", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)

planet_business_summary = universe_planets[["planet_key", "name", "population", "diameter", "climate", "terrain", "resident_keys", "resident_count", "film_keys", "film_count"]].copy().sort_values(["film_count", "resident_count", "population"], ascending=False)
starship_business_summary = universe_starships[["starship_key", "name", "starship_class", "manufacturer", "cost_in_credits", "length", "crew", "passengers", "cargo_capacity", "pilot_keys", "pilot_count", "film_keys", "film_count"]].copy().sort_values(["film_count", "pilot_count", "cost_in_credits"], ascending=False)
weapon_business_summary = universe_weapons[["weapon_key", "name", "type", "manufacturer", "cost_in_credits", "length", "film_keys", "film_count"]].copy().sort_values(["film_count", "cost_in_credits"], ascending=False)

story_featured_assets = pd.DataFrame([
    {"story_line": "Planetas como experiencias", "asset_name": "Tatooine", "asset_type": "planet", "story_role": "aventura, origen, desierto y nostalgia", "experience_concept": "Pop-up retro o ruta de inicio"},
    {"story_line": "Planetas como experiencias", "asset_name": "Hoth", "asset_type": "planet", "story_role": "batalla, supervivencia y accion", "experience_concept": "Experiencia inmersiva de mision"},
    {"story_line": "Planetas como experiencias", "asset_name": "Dagobah", "asset_type": "planet", "story_role": "entrenamiento Jedi, misterio y sabiduria", "experience_concept": "Escape room o entrenamiento"},
    {"story_line": "Planetas como experiencias", "asset_name": "Coruscant", "asset_type": "planet", "story_role": "ciudad futurista, tecnologia y escala", "experience_concept": "Instalacion tecnologica"},
    {"story_line": "Planetas como experiencias", "asset_name": "Naboo", "asset_type": "planet", "story_role": "estetica visual, elegancia y lifestyle", "experience_concept": "Colaboracion lifestyle"},
    {"story_line": "Planetas como experiencias", "asset_name": "Endor", "asset_type": "planet", "story_role": "naturaleza, comunidad y aventura familiar", "experience_concept": "Evento familiar y exterior"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "Millennium Falcon", "asset_type": "starship", "story_role": "aventura, libertad y silueta reconocible", "experience_concept": "Pieza central fotografiable"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "X-wing", "asset_type": "starship", "story_role": "Alianza Rebelde, velocidad y coleccionismo", "experience_concept": "Gaming, maquetas y accion"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "Lightsaber", "asset_type": "weapon", "story_role": "simbolo transversal de eleccion y poder", "experience_concept": "Activacion Choose Your Side"},
])

strategy_planet_experiences = story_featured_assets.query("asset_type == 'planet'").rename(columns={"asset_name": "planet_name", "story_role": "brand_atmosphere"})[["planet_name", "brand_atmosphere", "experience_concept"]]

display(universe_overview)
display(story_featured_assets)


In [ ]:
# Paginas 2, 3 y 7: Segmentos, emociones y rutas finales

AUDIENCE_ORDER = {
    "Jedi fiel": 1,
    "Rebelde nostalgico": 2,
    "Explorador casual": 3,
    "Territorio neutral": 4,
}


def classify_audience(row):
    is_fan = row.get("is_star_wars_fan_binary") == 1
    movies_seen = row.get("total_movies_seen", 0)
    if is_fan and movies_seen >= 5:
        return "Jedi fiel"
    if is_fan and movies_seen < 5:
        return "Rebelde nostalgico"
    if not is_fan and movies_seen >= 2:
        return "Explorador casual"
    return "Territorio neutral"


strategy_survey_respondents = survey_respondents.copy()
strategy_survey_respondents["audience_type"] = strategy_survey_respondents.apply(classify_audience, axis=1)
strategy_survey_respondents["audience_type_order"] = strategy_survey_respondents["audience_type"].map(AUDIENCE_ORDER)

total_respondents = len(strategy_survey_respondents)
strategy_audience_segments = (
    strategy_survey_respondents.groupby(["audience_type", "audience_type_order"], dropna=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        avg_movies_seen=("total_movies_seen", "mean"),
        fan_rate_pct=("is_star_wars_fan_binary", "mean"),
        avg_character_affinity=("average_character_opinion_score", "mean"),
        with_demographic_info_pct=("has_demographic_info", "mean"),
    )
    .reset_index()
    .sort_values("audience_type_order")
)
strategy_audience_segments["share_pct"] = strategy_audience_segments["respondents"] / total_respondents * 100
strategy_audience_segments["fan_rate_pct"] = strategy_audience_segments["fan_rate_pct"] * 100
strategy_audience_segments["with_demographic_info_pct"] = strategy_audience_segments["with_demographic_info_pct"] * 100
strategy_audience_segments["strategic_role"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Profundidad, lore, inmersion y pertenencia.",
    "Rebelde nostalgico": "Nostalgia, personajes clasicos y memoria emocional.",
    "Explorador casual": "Reconocimiento visual, iconos simples y acceso rapido.",
    "Territorio neutral": "Entrada ligera, digital y sin dependencia del lore.",
})
strategy_audience_segments["recommended_message"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Entra en la mision completa.",
    "Rebelde nostalgico": "Vuelve al momento que encendio la saga.",
    "Explorador casual": "Reconoce los iconos y elige tu lado.",
    "Territorio neutral": "Descubre Star Wars por sus simbolos universales.",
})
strategy_audience_segments["activation_goal"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Fidelizar y convertir en prescriptor.",
    "Rebelde nostalgico": "Reactivar recuerdo y compra emocional.",
    "Explorador casual": "Reducir friccion de entrada.",
    "Territorio neutral": "Crear primer contacto reconocible.",
})
strategy_audience_segments = strategy_audience_segments[[
    "audience_type_order", "audience_type", "respondents", "share_pct", "avg_movies_seen", "fan_rate_pct",
    "avg_character_affinity", "with_demographic_info_pct", "strategic_role", "recommended_message", "activation_goal"
]].round(2)

strategy_audience_age_matrix = (
    strategy_survey_respondents.groupby(["audience_type_order", "audience_type", "age"], dropna=False)
    .agg(respondents=("respondent_id", "nunique"), avg_movies_seen=("total_movies_seen", "mean"), fan_rate_pct=("is_star_wars_fan_binary", "mean"))
    .reset_index()
    .sort_values(["audience_type_order", "age"])
)
strategy_audience_age_matrix["fan_rate_pct"] = strategy_audience_age_matrix["fan_rate_pct"] * 100

emotion_map = {
    "luke_skywalker": ("Esperanza", "Heroe aspiracional", "Activaciones heroicas y mensajes de superacion."),
    "leia_organa": ("Liderazgo", "Autoridad rebelde", "Campanas de liderazgo, comunidad y resistencia."),
    "han_solo": ("Rebeldia", "Carisma inconformista", "Tono aventurero, nostalgico y de alta afinidad."),
    "yoda": ("Sabiduria", "Mentor Jedi", "Experiencias de aprendizaje, misterio y entrenamiento."),
    "obi_wan_kenobi": ("Legado", "Puente generacional", "Narrativa clasica con autoridad y memoria."),
    "darth_vader": ("Poder", "Icono premium polarizante", "Linea visual adulta, intensa y de alto reconocimiento."),
    "anakin_skywalker": ("Conflicto", "Transformacion", "Relatos de dualidad y eleccion de bando."),
    "r2_d2": ("Compania", "Humor y ternura", "Entrada amable para publico familiar y casual."),
    "c_3po": ("Humor", "Compania reconocible", "Activaciones ligeras, familiares y nostalgicas."),
    "emperor_palpatine": ("Amenaza", "Riesgo dramatico", "Contrapunto del Lado Oscuro, uso secundario."),
    "jar_jar_binks": ("Riesgo de rechazo", "Personaje polarizante", "Usar solo como aprendizaje de sesgo y tono."),
    "boba_fett": ("Misterio", "Coleccionismo", "Linea nicho para fans de iconografia y armaduras."),
    "padme_amidala": ("Elegancia", "Politica y estilo", "Activaciones lifestyle y estetica Naboo."),
    "lando_calrissian": ("Estilo", "Carisma secundario", "Campanas de nostalgia, moda y diferenciacion."),
}

strategy_character_emotional_map = character_opportunities.copy()
strategy_character_emotional_map["brand_emotion"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[0])
strategy_character_emotional_map["emotional_role"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[1])
strategy_character_emotional_map["activation_use"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[2])
strategy_character_emotional_map["polarization_risk"] = strategy_character_emotional_map["unfavorable_pct"].apply(lambda value: "alto" if value >= 25 else "medio" if value >= 10 else "bajo")
strategy_character_emotional_map = strategy_character_emotional_map[[
    "character_key", "character_name", "brand_emotion", "emotional_role", "activation_use", "familiarity_score",
    "audience_affinity_score", "quote_count", "favorable_pct", "unfavorable_pct", "opinion_responses",
    "opportunity_quadrant", "polarization_risk"
]].sort_values(["audience_affinity_score", "familiarity_score"], ascending=False)

strategy_experience_routes = pd.DataFrame([
    {"audience_type_order": 1, "audience_type": "Jedi fiel", "entry_gate": "The Empire Strikes Back", "key_characters": "Yoda, Luke Skywalker, Darth Vader", "world": "Dagobah / Hoth", "experience_format": "Experiencia inmersiva y mision por niveles", "primary_emotion": "Profundidad y pertenencia", "recommended_action": "Activar lore, retos, coleccionismo y contenido desbloqueable."},
    {"audience_type_order": 2, "audience_type": "Rebelde nostalgico", "entry_gate": "Trilogia original", "key_characters": "Han Solo, Leia Organa, Obi-Wan Kenobi", "world": "Tatooine", "experience_format": "Campana emocional retro", "primary_emotion": "Nostalgia y rebeldia", "recommended_action": "Usar escenas, frases y estetica clasica con baja complejidad."},
    {"audience_type_order": 3, "audience_type": "Explorador casual", "entry_gate": "Iconos reconocibles", "key_characters": "Darth Vader, Yoda, R2-D2", "world": "Escenarios reconocibles", "experience_format": "Activacion visual sencilla", "primary_emotion": "Reconocimiento inmediato", "recommended_action": "Priorizar simbolos, videos cortos, piezas sociales y rutas simples."},
    {"audience_type_order": 4, "audience_type": "Territorio neutral", "entry_gate": "Lightsaber y naves", "key_characters": "Simbolos antes que personajes", "world": "Universo simplificado", "experience_format": "Activacion digital de entrada", "primary_emotion": "Curiosidad", "recommended_action": "Evitar exceso de lore y crear una primera interaccion rapida."},
])

campaign_lines = strategy_experience_routes.rename(columns={
    "audience_type": "campaign_line",
    "key_characters": "main_assets",
    "experience_format": "recommended_products",
    "recommended_action": "business_reading",
}).copy()
campaign_lines["target"] = campaign_lines["campaign_line"]
campaign_lines = campaign_lines[["campaign_line", "main_assets", "recommended_products", "target", "business_reading", "entry_gate", "world", "primary_emotion"]]

storytelling_pages = pd.DataFrame([
    {"page_order": 1, "page_name": "La senal perdida", "business_question": "Donde sigue viva la conexion con Star Wars?", "main_tables": "eda_survey_kpis, eda_fan_by_age, eda_fan_by_gender, survey_respondents", "main_visuals": "KPIs, barras por edad/genero, segmentador fan_segment"},
    {"page_order": 2, "page_name": "Los clanes de la galaxia", "business_question": "Que tipos de audiencia necesita reactivar la marca?", "main_tables": "strategy_audience_segments, strategy_audience_age_matrix, strategy_survey_respondents", "main_visuals": "donut o barras de segmentos, matriz segmento x edad, promedio peliculas vistas"},
    {"page_order": 3, "page_name": "El mapa emocional de Star Wars", "business_question": "Que emocion de marca activa cada personaje?", "main_tables": "strategy_character_emotional_map, eda_character_merchandising_opportunities, eda_quote_character_summary", "main_visuals": "dispersion afinidad/familiaridad, amor vs rechazo, frases por personaje"},
    {"page_order": 4, "page_name": "Las puertas de entrada al universo", "business_question": "Que pelicula abre mejor la conversacion con cada publico?", "main_tables": "eda_movie_opportunities, eda_movie_commercial_audience_summary, films_business_clean", "main_visuals": "score pelicula, popularidad vs preferencia, impacto comercial vs conexion"},
    {"page_order": 5, "page_name": "Planetas como experiencias", "business_question": "Que atmosfera debe vivir cada publico?", "main_tables": "strategy_planet_experiences, eda_planet_business_summary, universe_planets_clean", "main_visuals": "tarjetas de experiencia, film_count, resident_count, mapa de mundos"},
    {"page_order": 6, "page_name": "Tecnologia, poder y velocidad", "business_question": "Que activos generan impacto visual y accion?", "main_tables": "eda_starship_business_summary, eda_weapon_business_summary, universe_starships_clean, universe_weapons_clean", "main_visuals": "naves por presencia, clase de nave, coste vs presencia, armas iconicas"},
    {"page_order": 7, "page_name": "La estrategia de reactivacion", "business_question": "Como convertir los datos en rutas de experiencia?", "main_tables": "strategy_experience_routes, story_campaign_lines, eda_conclusions, eda_survey_bias_visual", "main_visuals": "matriz final por audiencia, tarjetas de ruta, sesgos clave"},
])

display(strategy_audience_segments)
display(strategy_character_emotional_map.head(10))
display(strategy_experience_routes)


In [ ]:
# Calidad, sesgos y checks de relaciones

survey_missing_top = missing_report(survey_respondents).reset_index().rename(columns={"index": "column"})
survey_missing_top["dataset"] = "survey_respondents"

universe_missing_long = []
for dataset_name, df in {
    "characters": universe_characters,
    "films": universe_films,
    "planets": universe_planets,
    "starships": universe_starships,
    "weapons": universe_weapons,
    "quotes": universe_quotes,
}.items():
    temp = missing_report(df).reset_index().rename(columns={"index": "column"})
    temp["dataset"] = dataset_name
    universe_missing_long.append(temp)

universe_missing_long = pd.concat(universe_missing_long, ignore_index=True) if universe_missing_long else pd.DataFrame(columns=["column", "nulos", "porcentaje", "dataset"])
governance_missing_top = pd.concat([survey_missing_top[["dataset", "column", "nulos", "porcentaje"]], universe_missing_long[["dataset", "column", "nulos", "porcentaje"]]], ignore_index=True).sort_values(["porcentaje", "nulos"], ascending=False)

survey_sample_bias = pd.DataFrame([
    {"risk": "Muestra orientada a personas que conocen Star Wars", "evidence": f"{survey_kpis.loc[0, 'seen_any_star_wars_pct']}% declara haber visto alguna pelicula.", "business_impact": "Puede sobreestimar la demanda real del publico general."},
    {"risk": "Sesgo fan", "evidence": f"{survey_kpis.loc[0, 'star_wars_fan_pct']}% se declara fan entre respuestas validas.", "business_impact": "Las preferencias pueden favorecer personajes iconicos frente a nichos de crecimiento."},
    {"risk": "Datos demograficos incompletos", "evidence": f"{survey_kpis.loc[0, 'with_demographic_info_pct']}% tiene alguna informacion demografica util.", "business_impact": "La segmentacion por edad, genero, ingresos o region debe interpretarse con cautela."},
    {"risk": "Contenido expandido sin dimension propia", "evidence": "Algunas listas contienen series, videojuegos o personajes secundarios sin tabla maestra.", "business_impact": "No debe forzarse una relacion si el dashboard no analiza ese universo expandido."},
])

quality_checks = []
def add_quality_check(check_name, status, affected_rows, detail):
    quality_checks.append({"check_name": check_name, "status": status, "affected_rows": int(affected_rows), "detail": detail})

survey_character_keys = set(survey_character_opinions["character_key"].dropna().unique())
universe_character_keys = set(universe_characters["character_key"].dropna().unique())
survey_film_keys = set(survey_movies_seen["film_key"].dropna().unique()) | set(survey_movie_rankings["film_key"].dropna().unique())
universe_film_keys = set(universe_films["film_key"].dropna().unique())
business_film_keys = set(films_business_clean["film_key"].dropna().unique())

for name, missing in [
    ("survey_characters_exist_in_universe", sorted(survey_character_keys - universe_character_keys)),
    ("survey_films_exist_in_universe_films", sorted(survey_film_keys - universe_film_keys)),
    ("survey_films_exist_in_business_films", sorted(survey_film_keys - business_film_keys)),
]:
    add_quality_check(name, "pass" if not missing else "fail", len(missing), ", ".join(missing) if missing else "OK")

for check_name, df, column, allowed_keys in [
    ("planet_resident_keys_in_characters", universe_planets, "resident_keys", universe_character_keys),
    ("starship_pilot_keys_in_characters", universe_starships, "pilot_keys", universe_character_keys),
    ("vehicle_pilot_keys_in_characters", pd.read_csv(PROCESSED_DIR / "universe_vehicles_clean.csv"), "pilot_keys", universe_character_keys),
    ("planet_film_keys_in_films", universe_planets, "film_keys", universe_film_keys | {"all_episodes"}),
    ("starship_film_keys_in_films", universe_starships, "film_keys", universe_film_keys | {"all_episodes"}),
    ("weapon_film_keys_in_films", universe_weapons, "film_keys", universe_film_keys | {"all_episodes"}),
]:
    missing = sorted(collect_list_keys(df, column) - allowed_keys)
    add_quality_check(check_name, "pass" if not missing else "warn", len(missing), ", ".join(missing[:30]) if missing else "OK")

relationship_quality_checks = pd.DataFrame(quality_checks)
display(governance_missing_top.head(20))
display(survey_sample_bias)
display(relationship_quality_checks)


In [ ]:
# Conclusiones ejecutivas

best_movie_by_rank = eda_movie_commercial_audience_summary.sort_values("preference_score", ascending=False).iloc[0]
top_segment = strategy_audience_segments.sort_values("respondents", ascending=False).iloc[0]
top_character_emotion = strategy_character_emotional_map.sort_values("audience_affinity_score", ascending=False).iloc[0]

eda_conclusions = pd.DataFrame([
    {"area": "Audiencia", "finding": f"{survey_kpis.iloc[0]['seen_any_star_wars_pct']:.2f}% ha visto alguna pelicula y {survey_kpis.iloc[0]['star_wars_fan_pct']:.2f}% se declara fan.", "business_reading": "La marca tiene reconocimiento, pero la estrategia debe diferenciar niveles de vinculo."},
    {"area": "Segmentacion", "finding": f"El segmento mas grande es {top_segment['audience_type']} con {int(top_segment['respondents'])} respuestas.", "business_reading": "El dashboard debe hablar de clanes de audiencia, no de una masa unica."},
    {"area": "Peliculas", "finding": f"{best_movie_by_rank['movie_title']} lidera la conexion de audiencia con preference_score {best_movie_by_rank['preference_score']:.2f}.", "business_reading": "La pelicula ganadora funciona como puerta emocional, no solo como dato comercial."},
    {"area": "Personajes", "finding": f"{top_character_emotion['character_name']} destaca por afinidad y activa la emocion {top_character_emotion['brand_emotion']}.", "business_reading": "Los personajes deben leerse como emociones de marca, no solo como productos."},
    {"area": "Experiencias", "finding": "Tatooine, Hoth, Dagobah, Coruscant, Naboo y Endor representan atmosferas de campana distintas.", "business_reading": "Elegir un planeta equivale a elegir la sensacion que vivira el publico."},
    {"area": "Activos visuales", "finding": "Millennium Falcon, X-wing y Lightsaber concentran reconocimiento visual y accion.", "business_reading": "Los simbolos convierten la estrategia en una experiencia memorable y facil de comunicar."},
    {"area": "Recomendacion", "finding": "La propuesta final es Choose Your Side: una estrategia modular por tipo de audiencia.", "business_reading": "Star Wars no necesita una unica campana; necesita varias rutas de conexion bajo una misma marca."},
    {"area": "Sesgos", "finding": "La encuesta esta inclinada hacia personas familiarizadas con Star Wars.", "business_reading": "Las conclusiones son direccionales y no deben venderse como prediccion exacta del publico general."},
])

display(eda_conclusions)


In [ ]:
# Exportacion de resultados EDA y storytelling

exports = {
    "eda_survey_kpis.csv": survey_kpis,
    "eda_fan_by_age.csv": fan_by_age,
    "eda_fan_by_gender.csv": fan_by_gender,
    "eda_movie_views_summary.csv": movie_views_summary,
    "eda_movie_rank_summary.csv": movie_rank_summary,
    "eda_movie_opportunities.csv": movie_opportunities,
    "eda_movie_commercial_audience_summary.csv": eda_movie_commercial_audience_summary,
    "eda_character_opinion_summary.csv": character_opinion_summary,
    "eda_quote_character_summary.csv": quote_character_summary,
    "eda_character_merchandising_opportunities.csv": character_opportunities,
    "eda_universe_overview.csv": universe_overview,
    "eda_character_species_summary.csv": character_species_summary,
    "eda_character_gender_summary.csv": character_gender_summary,
    "eda_character_homeworld_summary.csv": character_homeworld_summary,
    "eda_planet_business_summary.csv": planet_business_summary,
    "eda_starship_business_summary.csv": starship_business_summary,
    "eda_weapon_business_summary.csv": weapon_business_summary,
    "eda_governance_missing_top.csv": governance_missing_top,
    "eda_survey_sample_bias.csv": survey_sample_bias,
    "eda_relationship_quality_checks.csv": relationship_quality_checks,
    "eda_conclusions.csv": eda_conclusions,
    "story_featured_assets.csv": story_featured_assets,
    "story_campaign_lines.csv": campaign_lines,
    "storytelling_powerbi_pages.csv": storytelling_pages,
    "strategy_audience_segments.csv": strategy_audience_segments,
    "strategy_audience_age_matrix.csv": strategy_audience_age_matrix,
    "strategy_survey_respondents.csv": strategy_survey_respondents,
    "strategy_character_emotional_map.csv": strategy_character_emotional_map,
    "strategy_planet_experiences.csv": strategy_planet_experiences,
    "strategy_experience_routes.csv": strategy_experience_routes,
}

for filename, df in exports.items():
    export_csv(df, filename)

if (relationship_quality_checks["status"] == "fail").any():
    raise ValueError("Hay checks criticos fallidos. Revisar eda_relationship_quality_checks.csv")
